# Config

In [1]:
setup_conda_env = True
install_docking_tools = True


# Filepaths

In [2]:
conda_env_folder = "conda_envs"

mgltools_folder = "prep_tools/MGLToolsPckgs"
autodock_folder = "docking_tools/autodock"
autodock_gpu_folder = "docking_tools/autodock_gpu"
diffdock_folder = "docking_tools/diffdock"
equibind_folder = "docking_tools/equibind"

receptor_folder = "Data/Receptors"
ligand_folder = "Data/Ligands/JKU"    

# Imports

In [3]:
import sys
from pathlib import Path

# Setup Conda Environments

# Benchmark Set || Autodock

In [4]:
import yaml, json, time, threading, shutil
from pathlib import Path
from datetime import datetime

sys.path.insert(0, str(Path.cwd()))

from Scripts.Docking.run_autodock import (
    run_autodock_vina, build_prepared_manifest, generate_summary,
    print_summary, get_cpu_model, precompute_properties,
    collect_files, get_pdbqt_dir, DockingResult,
)
from Scripts.Utilities.prep_docking import run_workflow, _write_box_file

# ── Paths ────────────────────────────────────────────────────────────────────
benchmark_dir = Path("Data/PoseBuster Benchmark Set")
output_base   = Path("Dockings/Benchmark")
log_dir       = Path("Dockings/Logs/benchmark_logs")
config_path   = Path("Scripts/Docking/autodock_vina_docking_config.yaml")

# ── Load base config and override output paths ──────────────────────────────
with open(config_path) as f:
    base_cfg = yaml.safe_load(f)

base_cfg["output_dir"] = str(output_base)
base_cfg["log_dir"]    = str(log_dir)

output_base.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

cpu_model = get_cpu_model()
print(f"CPU: {cpu_model}")

# ── Determine converter(s) from config ───────────────────────────────────────
prep_tool = base_cfg.get("prep_tool", "mgltools")
converters: list[tuple[str, str]] = []
if prep_tool in ("mgltools", "both"):
    converters.append(("mgl_tools", "mgltools"))
if prep_tool in ("meeko", "both"):
    converters.append(("meeko", "meeko"))
print(f"Converters: {[c[0] for c in converters]}")


def box_from_sdf(sdf_path: Path, padding: float = 10.0):
    """Compute docking-box center & size from an SDF ligand (crystal pose)."""
    from rdkit import Chem
    suppl = Chem.SDMolSupplier(str(sdf_path), removeHs=False)
    mol = next(iter(suppl))
    if mol is None:
        raise ValueError(f"Could not read molecule from {sdf_path}")
    conf = mol.GetConformer()
    xs, ys, zs = [], [], []
    for i in range(mol.GetNumAtoms()):
        pos = conf.GetAtomPosition(i)
        xs.append(pos.x); ys.append(pos.y); zs.append(pos.z)
    center = ((max(xs) + min(xs)) / 2, (max(ys) + min(ys)) / 2, (max(zs) + min(zs)) / 2)
    size   = ((max(xs) - min(xs)) + padding, (max(ys) - min(ys)) + padding, (max(zs) - min(zs)) + padding)
    return center, size


# ── Discover benchmark complexes ─────────────────────────────────────────────
complex_dirs = sorted(
    d for d in benchmark_dir.iterdir()
    if d.is_dir() and not d.name.startswith(("_", "."))
)
print(f"Found {len(complex_dirs)} benchmark complexes\n")

# ── Main loop ────────────────────────────────────────────────────────────────
all_results: list[DockingResult] = []
skipped, failed_prep = 0, 0

for idx, cdir in enumerate(complex_dirs, 1):
    pdb_id         = cdir.name                                     # e.g. "5S8I_2LY"
    protein_pdb    = cdir / f"{pdb_id}_protein.pdb"
    ligand_crystal = cdir / f"{pdb_id}_ligand.sdf"                 # crystal pose → defines box
    ligand_start   = cdir / f"{pdb_id}_ligand_start_conf.sdf"      # generated conf → dock this

    # ── Skip if files missing ────────────────────────────────────────────────
    if not protein_pdb.exists() or not ligand_start.exists() or not ligand_crystal.exists():
        print(f"[{idx}/{len(complex_dirs)}] SKIP {pdb_id} — missing files")
        skipped += 1
        continue

    # ── Skip if already docked (all converters) ──────────────────────────────
    done_markers = [output_base / pdb_id / cn / "docking_summary.json" for cn, _ in converters]
    if all(m.exists() for m in done_markers):
        print(f"[{idx}/{len(complex_dirs)}] ✓ {pdb_id} — already docked, skipping")
        skipped += 1
        continue

    print(f"\n{'─' * 60}")
    print(f"[{idx}/{len(complex_dirs)}] {pdb_id}")
    print(f"{'─' * 60}")

    # ── Create staging directories (one receptor, one ligand) ────────────────
    staging     = output_base / pdb_id / "_staging"
    rec_staging = staging / "receptors"
    lig_staging = staging / "ligands"
    rec_staging.mkdir(parents=True, exist_ok=True)
    lig_staging.mkdir(parents=True, exist_ok=True)

    # Symlink source files into staging (avoids copies)
    rec_link = rec_staging / protein_pdb.name
    lig_link = lig_staging / ligand_start.name
    if not rec_link.exists():
        rec_link.symlink_to(protein_pdb.resolve())
    if not lig_link.exists():
        lig_link.symlink_to(ligand_start.resolve())

    # ── Compute docking box from crystal ligand (10 Å padding) ───────────────
    try:
        center, size = box_from_sdf(ligand_crystal, padding=10.0)
    except Exception as exc:
        print(f"  ✗ Box computation failed: {exc}")
        failed_prep += 1
        continue

    # ── Iterate over converters ──────────────────────────────────────────────
    for conv_name, conv_arg in converters:
        vina_out = output_base / pdb_id / conv_name
        if (vina_out / "docking_summary.json").exists():
            print(f"  ✓ [{conv_name}] already docked — skipping")
            continue

        # ── Prepare protein PDBQT + box ──────────────────────────────────────
        protein_pdbqt_dir = get_pdbqt_dir(rec_staging)
        protein_outputs = run_workflow(
            input_dir=rec_staging,
            contains="proteins",
            output_dir=protein_pdbqt_dir,
            skip_pdb_validation=base_cfg.get("skip_pdb_validation", False),
            custom_postfix=f"_{conv_name}",
            process_postfixes=base_cfg.get("process_postfixes", False),
            repair_terminals=base_cfg.get("repair_terminals", False),
            converter=conv_arg,
            convert_proteins=True,
            verbose=False,
        )

        # Overwrite auto-generated box files with ligand-centered box
        for box_file in protein_pdbqt_dir.glob("*.box.txt"):
            _write_box_file(box_file, center, size)

        # ── Prepare ligand PDBQT (Meeko) ────────────────────────────────────
        ligand_pdbqt_dir = get_pdbqt_dir(lig_staging)
        ligand_outputs = run_workflow(
            input_dir=lig_staging,
            contains="ligands",
            output_dir=ligand_pdbqt_dir,
            process_postfixes=False,
            convert_ligands_with_meeko=True,
            verbose=False,
        )

        # ── Build manifest and dock ──────────────────────────────────────────
        manifest = build_prepared_manifest(protein_outputs, ligand_outputs)
        n_prot = len(manifest["proteins"])
        n_lig  = len(manifest["ligands"])
        if n_prot == 0 or n_lig == 0:
            print(f"  ✗ [{conv_name}] manifest empty (proteins={n_prot}, ligands={n_lig})")
            failed_prep += 1
            continue

        vina_out.mkdir(parents=True, exist_ok=True)

        results_df, results = run_autodock_vina(
            base_dir=vina_out,
            log_dir=log_dir,
            prepared_manifest=manifest,
            cfg=base_cfg,
            cpu_model=cpu_model,
            protein_workflow_data=protein_outputs,
            ligand_workflow_data=ligand_outputs,
        )

        # Save per-complex summary
        summary = generate_summary(results, base_cfg)
        with open(vina_out / "docking_summary.json", "w") as f:
            json.dump(summary, f, indent=2, default=str)

        for r in results:
            icon = "✓" if r.status == "success" else "✗"
            aff  = f"{r.best_affinity:.2f}" if r.best_affinity else "N/A"
            print(f"  {icon} [{conv_name}] {r.num_poses} poses | best: {aff} kcal/mol")
        all_results.extend(results)

# ── Final summary ────────────────────────────────────────────────────────────
n_ok   = sum(1 for r in all_results if r.status == "success")
n_fail = sum(1 for r in all_results if r.status == "failed")
print(f"\n{'=' * 60}")
print(f"BENCHMARK COMPLETE")
print(f"  Docked:       {n_ok}")
print(f"  Failed dock:  {n_fail}")
print(f"  Skipped:      {skipped}")
print(f"  Failed prep:  {failed_prep}")
print(f"  Results dir:  {output_base}")
print(f"{'=' * 60}")


CPU: Intel(R) Core(TM) i9-14900HX
Converters: ['mgl_tools', 'meeko']
Found 428 benchmark complexes

[1/428] ✓ 5S8I_2LY — already docked, skipping
[2/428] ✓ 5SAK_ZRY — already docked, skipping
[3/428] ✓ 5SB2_1K2 — already docked, skipping
[4/428] ✓ 5SD5_HWI — already docked, skipping
[5/428] ✓ 5SIS_JSM — already docked, skipping
[6/428] ✓ 6M2B_EZO — already docked, skipping
[7/428] ✓ 6M73_FNR — already docked, skipping
[8/428] ✓ 6T88_MWQ — already docked, skipping
[9/428] ✓ 6TW5_9M2 — already docked, skipping
[10/428] ✓ 6TW7_NZB — already docked, skipping
[11/428] ✓ 6VS3_R6V — already docked, skipping
[12/428] ✓ 6VTA_AKN — already docked, skipping
[13/428] ✓ 6W59_SZD — already docked, skipping
[14/428] ✓ 6WTN_RXT — already docked, skipping
[15/428] ✓ 6X8D_ARA — already docked, skipping
[16/428] ✓ 6XAF_GDP — already docked, skipping
[17/428] ✓ 6XBO_5MC — already docked, skipping
[18/428] ✓ 6XCT_478 — already docked, skipping
[19/428] ✓ 6XG5_TOP — already docked, skipping
[20/428] ✓ 6XHT_

# Benchmark Set || DiffDock

In [ ]:
import yaml, json, time, threading
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from Scripts.Docking.run_diffdock import (
    run_diffdock, generate_summary, print_summary,
    get_gpu_model, collect_files, DockingResult,
)

# ── Paths ────────────────────────────────────────────────────────────────────
benchmark_dir = Path("Data/PoseBuster Benchmark Set")
output_base   = Path("Dockings/Benchmark_DiffDock")
log_dir       = Path("Dockings/Logs/benchmark_diffdock_logs")
config_path   = Path("Scripts/Docking/diffdock_docking_config.yaml")

# ── Load config and override output/log paths ───────────────────────────────
with open(config_path) as f:
    dd_cfg = yaml.safe_load(f)

dd_cfg["output_dir"] = str(output_base)
dd_cfg["log_dir"]    = str(log_dir)

output_base.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

gpu_model = get_gpu_model()
print(f"GPU: {gpu_model}")

# ── Discover benchmark complexes ─────────────────────────────────────────────
complex_dirs = sorted(
    d for d in benchmark_dir.iterdir()
    if d.is_dir() and not d.name.startswith(("_", "."))
)
print(f"Found {len(complex_dirs)} benchmark complexes\n")

# ── Main loop — one DiffDock call per complex ────────────────────────────────
all_results: list[DockingResult] = []
skipped, failed = 0, 0

for idx, cdir in enumerate(complex_dirs, 1):
    pdb_id      = cdir.name                                    # e.g. "5S8I_2LY"
    protein_pdb = cdir / f"{pdb_id}_protein.pdb"
    ligand_sdf  = cdir / f"{pdb_id}_ligand_start_conf.sdf"    # generated conf → dock this

    # ── Skip if files missing ────────────────────────────────────────────────
    if not protein_pdb.exists() or not ligand_sdf.exists():
        print(f"[{idx}/{len(complex_dirs)}] SKIP {pdb_id} — missing files")
        skipped += 1
        continue

    # ── Skip if already docked ───────────────────────────────────────────────
    complex_out = output_base / pdb_id
    summary_file = complex_out / "docking_summary.json"
    if summary_file.exists():
        print(f"[{idx}/{len(complex_dirs)}] ✓ {pdb_id} — already docked, skipping")
        skipped += 1
        continue

    print(f"\n{'─' * 60}")
    print(f"[{idx}/{len(complex_dirs)}] {pdb_id}")
    print(f"{'─' * 60}")

    complex_out.mkdir(parents=True, exist_ok=True)

    # Override per-complex config paths (single protein + single ligand)
    per_cfg = dict(dd_cfg)
    per_cfg["output_dir"] = str(complex_out)
    per_cfg["log_dir"]    = str(log_dir)
    per_cfg["overwrite_existing"]  = False
    per_cfg["overwrite_error_log"] = True

    try:
        results = run_diffdock(
            proteins=[protein_pdb],
            ligands=[ligand_sdf],
            output_dir=complex_out,
            cfg=per_cfg,
            gpu_model=gpu_model,
        )
    except Exception as exc:
        print(f"  ✗ DiffDock error: {exc}")
        failed += 1
        continue

    # Save per-complex summary
    summary = generate_summary(results, per_cfg)
    with open(summary_file, "w") as f:
        json.dump(summary, f, indent=2, default=str)

    for r in results:
        icon = "✓" if r.status == "success" else "✗"
        print(f"  {icon} {r.num_poses} poses ({r.elapsed_time:.1f}s)")
        if r.error_message:
            print(f"      {r.error_message[:150]}")
    all_results.extend(results)

# ── Final summary ────────────────────────────────────────────────────────────
n_ok   = sum(1 for r in all_results if r.status == "success")
n_fail = sum(1 for r in all_results if r.status == "failed")
n_skip_res = sum(1 for r in all_results if r.status == "skipped")
print(f"\n{'=' * 60}")
print(f"DIFFDOCK BENCHMARK COMPLETE")
print(f"  Successful:   {n_ok}")
print(f"  Failed dock:  {n_fail + failed}")
print(f"  Skipped:      {skipped + n_skip_res}")
print(f"  Results dir:  {output_base}")
print(f"{'=' * 60}")

GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Found 428 benchmark complexes

[1/428] ✓ 5S8I_2LY — already docked, skipping
[2/428] ✓ 5SAK_ZRY — already docked, skipping
[3/428] ✓ 5SB2_1K2 — already docked, skipping
[4/428] ✓ 5SD5_HWI — already docked, skipping
[5/428] ✓ 5SIS_JSM — already docked, skipping
[6/428] ✓ 6M2B_EZO — already docked, skipping
[7/428] ✓ 6M73_FNR — already docked, skipping
[8/428] ✓ 6T88_MWQ — already docked, skipping
[9/428] ✓ 6TW5_9M2 — already docked, skipping
[10/428] ✓ 6TW7_NZB — already docked, skipping
[11/428] ✓ 6VS3_R6V — already docked, skipping
[12/428] ✓ 6VTA_AKN — already docked, skipping
[13/428] ✓ 6W59_SZD — already docked, skipping
[14/428] ✓ 6WTN_RXT — already docked, skipping
[15/428] ✓ 6X8D_ARA — already docked, skipping
[16/428] ✓ 6XAF_GDP — already docked, skipping
[17/428] ✓ 6XBO_5MC — already docked, skipping
[18/428] ✓ 6XCT_478 — already docked, skipping
[19/428] ✓ 6XG5_TOP — already docked, skipping
[20/428] ✓ 6XHT_V2V — already docked, skippin

# Benchmark Set || Equibind

In [ ]:
import os, shutil, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Paths ────────────────────────────────────────────────────────────────────
benchmark_dir          = Path("Data/PoseBuster Benchmark Set")
fpocket_results_folder = Path("pocket_results/fpocket_results")
p2rank_results_folder  = Path("pocket_results/p2rank_results")
p2rank_input_folder    = Path("pocket_results/p2rank_inputs")  # space-free staging
fpocket_results_folder.mkdir(parents=True, exist_ok=True)
p2rank_results_folder.mkdir(parents=True, exist_ok=True)
p2rank_input_folder.mkdir(parents=True, exist_ok=True)

# ── Tool binaries ────────────────────────────────────────────────────────────
FPOCKET_BIN = str(Path.home() / "tools" / "fpocket" / "bin" / "fpocket")
P2RANK_DIR  = str(Path.home() / "tools" / "p2rank_2.5")
P2RANK_BIN  = str(Path(P2RANK_DIR) / "prank")
for b in (FPOCKET_BIN, P2RANK_BIN):
    if not Path(b).exists():
        raise FileNotFoundError(b)

# ── Collect benchmark protein PDBs ───────────────────────────────────────────
complex_dirs = sorted(
    d for d in benchmark_dir.iterdir()
    if d.is_dir() and not d.name.startswith(("_", "."))
)
proteins = []
for cdir in complex_dirs:
    pdb = cdir / f"{cdir.name}_protein.pdb"
    if pdb.exists():
        proteins.append(pdb)
print(f"Benchmark proteins: {len(proteins)}")


# ─────────────────────────────────────────────────────────────────────────────
# fpocket — safely parallelizable (each call writes into its own *_out dir)
# ─────────────────────────────────────────────────────────────────────────────
def run_fpocket(pdb_file: Path):
    name = pdb_file.stem  # e.g. "5S8I_2LY_protein"
    out_dir = fpocket_results_folder / f"{name}_out"
    if (out_dir / f"{name}_info.txt").exists():
        return name, True, "skipped"
    target_pdb = fpocket_results_folder / pdb_file.name
    if not target_pdb.exists():
        shutil.copy2(pdb_file, target_pdb)
    proc = subprocess.run(
        [FPOCKET_BIN, "-f", str(target_pdb),
         "-m", "3", "-i", "3.0", "-n", "10"],
        capture_output=True, text=True,
    )
    return name, proc.returncode == 0, (proc.stderr or proc.stdout or "")[-300:]


def run_fpocket_parallel(items, max_workers):
    print(f"\n{'-' * 60}\nfpocket: {len(items)} proteins (workers={max_workers})\n{'-' * 60}")
    n_ok = n_skip = n_fail = 0
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(run_fpocket, p): p for p in items}
        for i, fut in enumerate(as_completed(futs), 1):
            name, ok, msg = fut.result()
            if msg == "skipped":
                n_skip += 1
            elif ok:
                n_ok += 1
            else:
                n_fail += 1
                print(f"  FAIL {name}: {msg}")
            if i % 50 == 0 or i == len(items):
                print(f"  [{i}/{len(items)}] ok={n_ok} skip={n_skip} fail={n_fail}")
    print(f"  Done: ok={n_ok} skipped={n_skip} failed={n_fail}")


# ─────────────────────────────────────────────────────────────────────────────
# p2rank — single JVM call over a dataset list.
#   .ds files are whitespace-delimited, so paths must NOT contain spaces.
#   We symlink each PDB into pocket_results/p2rank_inputs/ (space-free),
#   then list those symlinks in the dataset file.
# ─────────────────────────────────────────────────────────────────────────────
def stage_p2rank_inputs(items):
    staged = []
    for p in items:
        link = p2rank_input_folder / p.name
        if not link.exists():
            try:
                link.symlink_to(p.resolve())
            except OSError:
                shutil.copy2(p, link)
        staged.append(link)
    return staged


def run_p2rank_dataset(items):
    todo = [
        p for p in items
        if not (p2rank_results_folder / f"{p.stem}.pdb_predictions.csv").exists()
    ]
    n_skip = len(items) - len(todo)
    print(f"\n{'-' * 60}\np2rank: {len(todo)} proteins to predict, {n_skip} already done\n{'-' * 60}")
    if not todo:
        return

    staged = stage_p2rank_inputs(todo)
    if any(" " in str(s.absolute()) for s in staged):
        raise RuntimeError("staged input path contains spaces — p2rank dataset cannot handle this")

    ds_file = p2rank_input_folder / "_benchmark.ds"
    ds_file.write_text(
        "HEADER: protein\n\n" +
        "\n".join(str(s.absolute()) for s in staged) + "\n"
    )

    n_threads = max(1, min(os.cpu_count() or 4, 8))
    cmd = [
        P2RANK_BIN, "predict",
        "-threads", str(n_threads),
        "-o", str(p2rank_results_folder.resolve()),
        str(ds_file.resolve()),
    ]
    print(f"  cmd: {' '.join(cmd)}")
    proc = subprocess.run(cmd, cwd=P2RANK_DIR, capture_output=True, text=True)

    if proc.returncode != 0:
        print(f"  ✗ p2rank exited with code {proc.returncode}")
        print("--- last 2KB stdout ---")
        print((proc.stdout or "")[-2000:])
        print("--- last 2KB stderr ---")
        print((proc.stderr or "")[-2000:])
    else:
        n_done = sum(
            1 for p in todo
            if (p2rank_results_folder / f"{p.stem}.pdb_predictions.csv").exists()
        )
        print(f"  ✓ p2rank done: {n_done}/{len(todo)} predictions written")


n_workers = max(1, min(os.cpu_count() or 8, 16))
run_fpocket_parallel(proteins, n_workers)
run_p2rank_dataset(proteins)

# ── Verify counts ────────────────────────────────────────────────────────────
n_fp = len(list(fpocket_results_folder.glob("*_out")))
n_p2 = len(list(p2rank_results_folder.glob("*.pdb_predictions.csv")))
print(f"\nfpocket _out dirs:        {n_fp}")
print(f"p2rank predictions CSVs:  {n_p2}")


Benchmark proteins: 428

------------------------------------------------------------
fpocket: 428 proteins (workers=16)
------------------------------------------------------------
  [50/428] ok=0 skip=50 fail=0
  [100/428] ok=0 skip=100 fail=0
  [150/428] ok=0 skip=150 fail=0
  [200/428] ok=0 skip=200 fail=0
  [250/428] ok=0 skip=250 fail=0
  [300/428] ok=0 skip=300 fail=0
  [350/428] ok=0 skip=350 fail=0
  [400/428] ok=0 skip=400 fail=0
  [428/428] ok=0 skip=428 fail=0
  Done: ok=0 skipped=428 failed=0

------------------------------------------------------------
p2rank: 428 proteins to predict, 0 already done
------------------------------------------------------------
  cmd: /home/manndo/tools/p2rank_2.5/prank predict -threads 8 -o /home/manndo/master_dev/pocket_results/p2rank_results /home/manndo/master_dev/pocket_results/p2rank_inputs/_benchmark.ds


In [ ]:
import os, sys, json, subprocess, yaml
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

# ── Paths ────────────────────────────────────────────────────────────────────
benchmark_dir = Path("Data/PoseBuster Benchmark Set")
output_base   = Path("Dockings/Benchmark_Equibind")
log_dir       = Path("Dockings/Logs/benchmark_equibind_logs")
config_path   = Path("Scripts/Docking/equibind_docking_config.yaml")
runner_script = Path("Scripts/Docking/run_equibind.py")
staging_root  = output_base / "_staging"

output_base.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)
staging_root.mkdir(parents=True, exist_ok=True)

# ── Load config (used only to source EquiBind dir / device / batch knobs) ───
with open(config_path) as f:
    eb_cfg = yaml.safe_load(f)

equibind_dir   = os.path.expanduser(eb_cfg.get("equibind_dir", "~/tools/EquiBind"))
device         = eb_cfg.get("device", "cuda")
gpu_batch_size = int(eb_cfg.get("gpu_batch_size", 8))

# Use EquiBind conda env's Python (notebook kernel lacks torch)
equibind_python = Path("/home/manndo/anaconda3/envs/equibind/bin/python")
if not equibind_python.exists():
    raise FileNotFoundError(f"EquiBind python not found: {equibind_python}")

print(f"EquiBind dir:    {equibind_dir}")
print(f"EquiBind python: {equibind_python}")
print(f"Device:          {device}")

# ── Discover benchmark complexes ─────────────────────────────────────────────
complex_dirs = sorted(
    d for d in benchmark_dir.iterdir()
    if d.is_dir() and not d.name.startswith(("_", "."))
)
print(f"Found {len(complex_dirs)} benchmark complexes\n")

# ── Main loop — one EquiBind invocation per complex ──────────────────────────
n_ok, n_fail, skipped = 0, 0, 0

for idx, cdir in enumerate(complex_dirs, 1):
    pdb_id      = cdir.name                                    # e.g. "5S8I_2LY"
    protein_pdb = cdir / f"{pdb_id}_protein.pdb"
    ligand_sdf  = cdir / f"{pdb_id}_ligand_start_conf.sdf"     # generated conf → dock this

    # ── Skip if files missing ────────────────────────────────────────────────
    if not protein_pdb.exists() or not ligand_sdf.exists():
        print(f"[{idx}/{len(complex_dirs)}] SKIP {pdb_id} — missing files")
        skipped += 1
        continue

    # ── Skip if already docked ───────────────────────────────────────────────
    complex_out  = output_base / pdb_id
    summary_file = complex_out / "pipeline_summary.json"
    if summary_file.exists():
        print(f"[{idx}/{len(complex_dirs)}] ✓ {pdb_id} — already docked, skipping")
        skipped += 1
        continue

    print(f"\n{'─' * 60}")
    print(f"[{idx}/{len(complex_dirs)}] {pdb_id}")
    print(f"{'─' * 60}")

    # ── Stage protein + ligand into flat per-complex dirs (symlinks) ─────────
    rec_staging = staging_root / pdb_id / "receptors"
    lig_staging = staging_root / pdb_id / "ligands"
    rec_staging.mkdir(parents=True, exist_ok=True)
    lig_staging.mkdir(parents=True, exist_ok=True)

    rec_link = rec_staging / protein_pdb.name
    lig_link = lig_staging / ligand_sdf.name
    if not rec_link.exists():
        rec_link.symlink_to(protein_pdb.resolve())
    if not lig_link.exists():
        lig_link.symlink_to(ligand_sdf.resolve())

    complex_out.mkdir(parents=True, exist_ok=True)

    # ── Invoke run_equibind.py with EQ_* env-var overrides ───────────────────
    env = os.environ.copy()
    env.update({
        "EQ_RECEPTORS_DIR":   str(rec_staging.resolve()),
        "EQ_DRUGS_DIR":       str(lig_staging.resolve()),
        "EQ_OUTPUT_DIR":      str(complex_out.resolve()),
        "EQ_RECEPTOR_FILTER": "",                                # match any *.pdb
        "EQ_EQUIBIND_DIR":    equibind_dir,
        "EQ_DEVICE":          device,
        "EQ_GPU_BATCH_SIZE":  str(gpu_batch_size),
        "EQ_FPOCKET_DIR":     str(Path(eb_cfg.get("fpocket_results_dir", "pocket_results/fpocket_results")).resolve()),
        "EQ_P2RANK_DIR":      str(Path(eb_cfg.get("p2rank_results_dir",  "pocket_results/p2rank_results")).resolve()),
    })

    log_file = log_dir / f"{pdb_id}.log"
    with open(log_file, "w") as lf:
        proc = subprocess.run(
            [str(equibind_python), str(runner_script.resolve())],
            env=env,
            cwd=str(Path.cwd()),
            stdout=lf,
            stderr=subprocess.STDOUT,
        )

    if proc.returncode == 0 and summary_file.exists():
        try:
            with open(summary_file) as f:
                s = json.load(f)
            ok  = s.get("totals", {}).get("poses_success", "?")
            bad = s.get("totals", {}).get("poses_failed",  "?")
            wt  = s.get("global_timing", {}).get("pipeline_wall_time_s", 0.0)
            print(f"  ✓ poses ok={ok} fail={bad}  ({wt:.1f}s)")
        except Exception:
            print(f"  ✓ done (summary unreadable)")
        n_ok += 1
    else:
        print(f"  ✗ EquiBind failed (rc={proc.returncode}) — see {log_file}")
        n_fail += 1

# ── Final summary ────────────────────────────────────────────────────────────
print(f"\n{'=' * 60}")
print(f"EQUIBIND BENCHMARK COMPLETE")
print(f"  Successful:   {n_ok}")
print(f"  Failed dock:  {n_fail}")
print(f"  Skipped:      {skipped}")
print(f"  Results dir:  {output_base}")
print(f"{'=' * 60}")

EquiBind dir:    /home/manndo/tools/EquiBind
EquiBind python: /home/manndo/anaconda3/envs/equibind/bin/python
Device:          cuda
Found 428 benchmark complexes


────────────────────────────────────────────────────────────
[1/428] 5S8I_2LY
────────────────────────────────────────────────────────────
  ✓ poses ok=5 fail=0  (4.3s)

────────────────────────────────────────────────────────────
[2/428] 5SAK_ZRY
────────────────────────────────────────────────────────────
  ✓ poses ok=5 fail=0  (4.2s)

────────────────────────────────────────────────────────────
[3/428] 5SB2_1K2
────────────────────────────────────────────────────────────
  ✓ poses ok=5 fail=0  (3.2s)

────────────────────────────────────────────────────────────
[4/428] 5SD5_HWI
────────────────────────────────────────────────────────────


KeyboardInterrupt: 

# Benchmark Set || PoseBusters validation (AutoDock + DiffDock + EquiBind)

Runs `Scripts/Docking/Posebusters/run_posebusters.py` on all benchmark
docking outputs.

The script's collectors expect a flat per-method layout:

```
<staging>/receptors/<pdb_id>_protein.pdb
<staging>/autodock/<pdb_id>__<pdb_id>_vina_out.pdbqt
<staging>/diffdock/<pdb_id>__<pdb_id>/*.sdf
<staging>/equibind/<pdb_id>__<pdb_id>/*.sdf
```

This cell symlinks the benchmark outputs into that layout, then invokes the
script with `posebusters_benchmark_config.yaml`.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

# ── Paths ────────────────────────────────────────────────────────────────────
benchmark_dir       = Path("Data/PoseBuster Benchmark Set")
autodock_results    = Path("Dockings/Benchmark")
diffdock_results    = Path("Dockings/Benchmark_DiffDock")
equibind_results    = Path("Dockings/Benchmark_Equibind")

pb_config_path      = Path("Scripts/Docking/Posebusters/posebusters_benchmark_config.yaml")
pb_runner_script    = Path("Scripts/Docking/Posebusters/run_posebusters.py")

staging_root        = Path("posebusters_results/_benchmark_staging")
rec_staging         = staging_root / "receptors"
ad_staging          = staging_root / "autodock"
dd_staging          = staging_root / "diffdock"
eb_staging          = staging_root / "equibind"

for d in (rec_staging, ad_staging, dd_staging, eb_staging):
    d.mkdir(parents=True, exist_ok=True)


def _link(src: Path, dst: Path) -> bool:
    """Create or refresh a symlink dst -> src. Returns True on success."""
    if not src.exists():
        return False
    if dst.is_symlink() or dst.exists():
        try:
            if dst.resolve() == src.resolve():
                return True
        except OSError:
            pass
        dst.unlink()
    try:
        dst.symlink_to(src.resolve())
    except OSError:
        shutil.copy2(src, dst)
    return True


# ── Discover benchmark complexes (one folder per PDB-ID) ────────────────────
complex_dirs = sorted(
    d for d in benchmark_dir.iterdir()
    if d.is_dir() and not d.name.startswith(("_", "."))
)
print(f"Benchmark complexes: {len(complex_dirs)}")

n_rec = n_ad = n_dd = n_eb = 0
missing_protein = []

# Staging filename convention (matches the collectors in run_posebusters.py):
#   autodock:  <protein>__<ligand>_vina_out.pdbqt  → split on "__" after stripping "_vina_out"
#   diffdock:  <ligand>__<protein>/                → split on "__"
#   equibind:  <ligand>__<protein>/                → split on "__"
#
# We use:
#   protein label = "<pdb_id>_protein"  (matches our staged receptor stem exactly)
#   ligand  label = "<pdb_id>_ligand"   (uniform, harmless label)

for cdir in complex_dirs:
    pdb_id      = cdir.name                       # e.g. "5S8I_2LY"
    protein_pdb = cdir / f"{pdb_id}_protein.pdb"

    # ── Receptor ─────────────────────────────────────────────────────────────
    if _link(protein_pdb, rec_staging / protein_pdb.name):
        n_rec += 1
    else:
        missing_protein.append(pdb_id)
        continue  # without a receptor the entry is useless

    p_label = f"{pdb_id}_protein"
    l_label = f"{pdb_id}_ligand"

    # ── AutoDock: prefer meeko, fall back to mgl_tools ───────────────────────
    # Real filename: <pdb_id>_protein_<conv>__<pdb_id>_ligand_start_conf_vina_vina_out.pdbqt
    ad_complex = autodock_results / pdb_id
    if ad_complex.is_dir():
        chosen = None
        for sub in ("meeko", "mgl_tools"):
            cand = ad_complex / sub
            if cand.is_dir():
                pdbqts = sorted(cand.rglob("*_vina_out.pdbqt"))
                if not pdbqts:
                    pdbqts = sorted(cand.rglob("*.pdbqt"))
                if pdbqts:
                    chosen = pdbqts[0]
                    break
        if chosen is not None:
            staged = ad_staging / f"{p_label}__{l_label}_vina_out.pdbqt"
            if _link(chosen, staged):
                n_ad += 1

    # ── DiffDock: link the inner output dir as <ligand>__<protein>/ ──────────
    # Real layout: Dockings/Benchmark_DiffDock/<pdb_id>/<pdb_id>_start_conf__<pdb_id>/*.sdf
    dd_complex = diffdock_results / pdb_id
    if dd_complex.is_dir():
        inner_dirs = [d for d in dd_complex.iterdir() if d.is_dir() and any(d.rglob("*.sdf"))]
        if inner_dirs:
            staged = dd_staging / f"{l_label}__{p_label}"
            if _link(inner_dirs[0], staged):
                n_dd += 1

    # ── EquiBind: link the inner output dir as <ligand>__<protein>/ ──────────
    # Real layout: Dockings/Benchmark_Equibind/<pdb_id>/<pdb_id>_ligand_start_conf__<pdb_id>_protein/*.sdf
    eb_complex = equibind_results / pdb_id
    if eb_complex.is_dir():
        inner_dirs = [
            d for d in eb_complex.iterdir()
            if d.is_dir() and "__" in d.name
            and any(f for f in d.rglob("*.sdf") if "prep" not in f.parts)
        ]
        if inner_dirs:
            staged = eb_staging / f"{l_label}__{p_label}"
            if _link(inner_dirs[0], staged):
                n_eb += 1

print(f"\nStaging summary:")
print(f"  receptors:  {n_rec:>4}  → {rec_staging}")
print(f"  autodock:   {n_ad:>4}  → {ad_staging}")
print(f"  diffdock:   {n_dd:>4}  → {dd_staging}")
print(f"  equibind:   {n_eb:>4}  → {eb_staging}")
if missing_protein:
    print(f"\n⚠ {len(missing_protein)} complexes lack a *_protein.pdb (skipped):")
    for pid in missing_protein[:10]:
        print(f"    • {pid}")
    if len(missing_protein) > 10:
        print(f"    … +{len(missing_protein) - 10} more")

# ── Sanity-check config + runner exist ──────────────────────────────────────
if not pb_config_path.exists():
    raise FileNotFoundError(pb_config_path)
if not pb_runner_script.exists():
    raise FileNotFoundError(pb_runner_script)

# ── Invoke run_posebusters.py with the benchmark config ─────────────────────
print(f"\n{'═' * 60}")
print(f"PoseBusters validation")
print(f"  config:  {pb_config_path}")
print(f"  runner:  {pb_runner_script}")
print(f"{'═' * 60}\n")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

proc = subprocess.run(
    [sys.executable, str(pb_runner_script.resolve()),
     "--config", str(pb_config_path.resolve())],
    cwd=str(Path.cwd()),
    env=env,
)

print(f"\n{'=' * 60}")
print(f"PoseBusters BENCHMARK COMPLETE  (return code: {proc.returncode})")
print(f"  Results: posebusters_results/benchmark/dock/")
print(f"  Plots:   posebusters_results/benchmark/dock/pb_*.png")
print(f"  Proved:  posebuster_proved/dock/")
print(f"{'=' * 60}")


# Orai Section

# Prepare and Create Orai Receptor Files (PDB)

In [ ]:
sys.path.insert(0, str(Path.cwd().parents[1])) 

# Batch clean directory
from Scripts.Utilities.prepare_receptor_pdb import batch_clean_pdbs
batch_clean_pdbs(f"{receptor_folder}/Original", pattern='*.pdb', output_dir=receptor_folder)

# Prepare Ligand Files (PDB)

In [ ]:
from Scripts.Utilities.prep_docking import run_workflow

ligand_outputs = run_workflow(
    input_dir=Path(ligand_folder),
    contains="ligands",
    ligand_formats=["pdb"],
    output_dir=Path(ligand_folder),
    process_postfixes=False,
)


# Autodock Vina

## Autodock Config

In [ ]:
import yaml

config_path = Path("Scripts/Docking/autodock_vina_docking_config.yaml")

# Load current config
with open(config_path, "r") as f:
    docking_cfg = yaml.safe_load(f)

# ── Edit any values below, then run this cell to save ────────────────────────

# Input paths (synced with notebook variables by default)
docking_cfg["receptors_dir"]       = receptor_folder          # "Data/Receptors"
docking_cfg["ligand_dirs"]         = [ligand_folder]           # ["Data/Ligands/JKU"]

# Preparation tools
docking_cfg["prep_tool"]           = "both"         # "mgltools", "meeko", or "both"
docking_cfg["skip_pdb_validation"] = False          # True=skip, False=validate (default: False)
docking_cfg["process_postfixes"]   = False         # True=add _prepared/_pdbqt postfixes, False=keep original names (default: False)
docking_cfg["repair_terminals"]    = False         # True=repair, False=do not repair (default: False)

# Output
docking_cfg["output_dir"]          = "Dockings/vina_results"
docking_cfg["log_dir"]             = "Dockings/Logs/vina_logs"

# Vina executable
docking_cfg["vina_bin"]            = "/home/manndo/AutoDock-Vina/build/linux/release/vina"

# Scoring function
docking_cfg["scoring_function"]    = "vina"         # "vina", "vinardo", or "ad4"
docking_cfg["autogrid_bin"]        = "/usr/local/bin/autogrid4"

# Batch mode
docking_cfg["batch_mode"]          = None            # None=auto, True=force, False=disable
docking_cfg["batch_size"]          = 10
docking_cfg["batch_timeout"]       = 900

# Docking parameters
docking_cfg["exhaustiveness"]      = 32
docking_cfg["num_modes"]           = 10
docking_cfg["energy_range"]        = 3
docking_cfg["seed"]                = 42
docking_cfg["timeout_per_complex"] = 600

# Parallelism
docking_cfg["max_workers"]         = 1
docking_cfg["cpus_per_worker"]     = 32

# Overwrite settings
docking_cfg["overwrite_existing"]  = True
docking_cfg["overwrite_poses"]     = True
docking_cfg["overwrite_error_log"] = True

# ── Save updated config ─────────────────────────────────────────────────────
with open(config_path, "w") as f:
    yaml.dump(docking_cfg, f, default_flow_style=False, sort_keys=False)

print(f"✓ Config saved to {config_path}")
print(yaml.dump(docking_cfg, default_flow_style=False, sort_keys=False))


In [ ]:
%run Scripts/Docking/run_autodock.py -c Scripts/Docking/autodock_vina_docking_config.yaml

# Orai × PoseBuster Benchmark Ligands || AutoDock Vina (batched)

Docks every Orai receptor in `Data/Receptors/*.pdb` against every ligand
(`*_ligand_start_conf.sdf`) in `Data/PoseBuster Benchmark Set/`,
running 50 ligands per Vina invocation per receptor.

In [ ]:
import yaml, json, shutil
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from Scripts.Docking.run_autodock import (
    run_autodock_vina, build_prepared_manifest, generate_summary,
    get_cpu_model, collect_files, get_pdbqt_dir, DockingResult,
)
from Scripts.Utilities.prep_docking import run_workflow

# ── Paths ────────────────────────────────────────────────────────────────────
benchmark_dir = Path("Data/PoseBuster Benchmark Set")
receptors_dir = Path(receptor_folder)                       # Data/Receptors
output_base   = Path("Dockings/Orai_Benchmark")
log_dir       = Path("Dockings/Logs/orai_benchmark_logs")
config_path   = Path("Scripts/Docking/orai_benchmark_autodock_config.yaml")
staging_root  = output_base / "_staging"
lig_staging   = staging_root / "ligands"
rec_staging   = staging_root / "receptors"

# ── Load + freeze the on-disk config (do not overwrite output_dir to disk) ──
with open(config_path) as f:
    cfg = yaml.safe_load(f)

cfg["receptors_dir"] = str(rec_staging)
cfg["ligand_dirs"]   = [str(lig_staging)]
cfg["output_dir"]    = str(output_base)
cfg["log_dir"]       = str(log_dir)
cfg["batch_mode"]    = True
cfg["batch_size"]    = 50

output_base.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)
lig_staging.mkdir(parents=True, exist_ok=True)
rec_staging.mkdir(parents=True, exist_ok=True)

cpu_model = get_cpu_model()
print(f"CPU: {cpu_model}")

# ── Stage receptors (symlink the 4 Orai PDBs into staging) ──────────────────
receptor_pdbs = sorted(p for p in receptors_dir.glob("*.pdb") if p.is_file())
if not receptor_pdbs:
    raise FileNotFoundError(f"No receptor PDBs found in {receptors_dir}")
for r in receptor_pdbs:
    link = rec_staging / r.name
    if not link.exists():
        link.symlink_to(r.resolve())
print(f"Receptors staged: {len(receptor_pdbs)}")
for r in receptor_pdbs:
    print(f"  • {r.name}")

# ── Stage benchmark ligands (symlink every *_ligand_start_conf.sdf) ─────────
complex_dirs = sorted(
    d for d in benchmark_dir.iterdir()
    if d.is_dir() and not d.name.startswith(("_", "."))
)
n_staged, n_missing = 0, 0
for cdir in complex_dirs:
    lig = cdir / f"{cdir.name}_ligand_start_conf.sdf"
    if not lig.exists():
        n_missing += 1
        continue
    link = lig_staging / lig.name
    if not link.exists():
        link.symlink_to(lig.resolve())
    n_staged += 1
print(f"\nBenchmark ligands staged: {n_staged}  (missing: {n_missing})")

# ── Prepare receptors → PDBQT + auto-generated whole-protein boxes ──────────
print(f"\n{'─' * 60}\nPreparing receptors (PDB → PDBQT + box)\n{'─' * 60}")
protein_pdbqt_dir = get_pdbqt_dir(rec_staging)
protein_outputs = run_workflow(
    input_dir=rec_staging,
    contains="proteins",
    output_dir=protein_pdbqt_dir,
    skip_pdb_validation=cfg.get("skip_pdb_validation", False),
    process_postfixes=cfg.get("process_postfixes", False),
    repair_terminals=cfg.get("repair_terminals", False),
    converter=cfg.get("prep_tool", "meeko"),
    convert_proteins=True,
    verbose=False,
)

# ── Prepare ligands → PDBQT (Meeko) ─────────────────────────────────────────
print(f"\n{'─' * 60}\nPreparing ligands (SDF → PDBQT via Meeko)\n{'─' * 60}")
ligand_pdbqt_dir = get_pdbqt_dir(lig_staging)
ligand_outputs = run_workflow(
    input_dir=lig_staging,
    contains="ligands",
    output_dir=ligand_pdbqt_dir,
    process_postfixes=False,
    convert_ligands_with_meeko=True,
    verbose=False,
)

# ── Build manifest ──────────────────────────────────────────────────────────
manifest = build_prepared_manifest(protein_outputs, ligand_outputs)
n_prot = len(manifest["proteins"])
n_lig  = len(manifest["ligands"])
print(f"\nManifest: {n_prot} prepared protein entries  |  {n_lig} prepared ligand entries")

if n_prot == 0 or n_lig == 0:
    raise RuntimeError("Empty manifest — preparation failed for proteins or ligands.")

# ── Run AutoDock Vina (batched: 50 ligands per Vina call per receptor) ──────
print(f"\n{'═' * 60}")
print(f"Docking {len(receptor_pdbs)} receptors × {n_staged} ligands "
      f"(batch_size={cfg['batch_size']})")
print(f"{'═' * 60}")

results_df, results = run_autodock_vina(
    base_dir=output_base,
    log_dir=log_dir,
    prepared_manifest=manifest,
    cfg=cfg,
    cpu_model=cpu_model,
    protein_workflow_data=protein_outputs,
    ligand_workflow_data=ligand_outputs,
)

# ── Per-receptor summaries ──────────────────────────────────────────────────
summary = generate_summary(results, cfg)
with open(output_base / "docking_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

n_ok   = sum(1 for r in results if r.status == "success")
n_fail = sum(1 for r in results if r.status == "failed")
n_skip = sum(1 for r in results if r.status == "skipped")

print(f"\n{'=' * 60}")
print(f"ORAI × BENCHMARK DOCKING COMPLETE")
print(f"  Receptors:    {len(receptor_pdbs)}")
print(f"  Ligands:      {n_staged}")
print(f"  Combinations: {len(results)}")
print(f"  Success:      {n_ok}")
print(f"  Failed:       {n_fail}")
print(f"  Skipped:      {n_skip}")
print(f"  Results dir:  {output_base}")
print(f"  Summary:      {output_base / 'docking_summary.json'}")
print(f"{'=' * 60}")

CPU: Intel(R) Core(TM) i9-14900HX
Receptors staged: 4
  • Orai1WT-MDSnap-Fr300.pdb
  • Orai1WT-MDSnap-Fr400.pdb
  • Orai1WT-MDSnap-Fr499.pdb
  • Orai1WT-START-Fr0.pdb

Benchmark ligands staged: 428  (missing: 0)

────────────────────────────────────────────────────────────
Preparing receptors (PDB → PDBQT + box)
────────────────────────────────────────────────────────────

PROCESSING 4 PROTEIN STRUCTURE(S)

────────────────────────────────────────────────────────────
Preparing ligands (SDF → PDBQT via Meeko)
────────────────────────────────────────────────────────────

Manifest: 12 prepared protein entries  |  856 prepared ligand entries

════════════════════════════════════════════════════════════
Docking 4 receptors × 428 ligands (batch_size=50)
════════════════════════════════════════════════════════════
Docking receptors: 4 | ligands: 428 | scoring: vina
Batch mode: ON (4 receptor(s), 428 ligands per Vina invocation)
Docking log: /home/manndo/master_dev/Dockings/Orai_Benchmark/dock

# Orai × PoseBuster Benchmark Ligands || DiffDock

Docks every Orai receptor in `Data/Receptors/*.pdb` (top level only) against
every ligand in `Data/Ligands/PoseBuster_Benchmark_Set/*.sdf` using DiffDock
in CSV-batch mode.

In [ ]:
import yaml, json
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from Scripts.Docking.run_diffdock import (
    run_diffdock, generate_summary, get_gpu_model, DockingResult,
)

# ── Paths ────────────────────────────────────────────────────────────────────
receptors_dir = Path("Data/Receptors")
ligands_dir   = Path("Data/Ligands/PoseBuster_Benchmark_Set")
output_base   = Path("Dockings/Orai_Benchmark_DiffDock")
log_dir       = Path("Dockings/Logs/orai_benchmark_diffdock_logs")
config_path   = Path("Scripts/Docking/diffdock_docking_config.yaml")

# ── Load DiffDock config and force batch mode ───────────────────────────────
with open(config_path) as f:
    dd_cfg = yaml.safe_load(f)

dd_cfg["output_dir"]         = str(output_base)
dd_cfg["log_dir"]            = str(log_dir)
dd_cfg["batch_mode"]         = True
dd_cfg["batch_size"]         = dd_cfg.get("batch_size", 15)
dd_cfg["overwrite_existing"] = False

output_base.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

gpu_model = get_gpu_model()
print(f"GPU: {gpu_model}")

# ── Collect Orai receptors (top-level *.pdb only — no subfolders) ───────────
receptor_pdbs = sorted(p for p in receptors_dir.glob("*.pdb") if p.is_file())
if not receptor_pdbs:
    raise FileNotFoundError(f"No receptor PDBs found in {receptors_dir}")
print(f"\nReceptors ({len(receptor_pdbs)}):")
for r in receptor_pdbs:
    print(f"  • {r.name}")

# ── Collect benchmark ligands ───────────────────────────────────────────────
ligand_sdfs = sorted(p for p in ligands_dir.glob("*.sdf") if p.is_file())
if not ligand_sdfs:
    raise FileNotFoundError(f"No ligand SDFs found in {ligands_dir}")
print(f"\nLigands: {len(ligand_sdfs)}")

# ── Run DiffDock (one CSV-batch invocation handles all combinations) ────────
print(f"\n{'═' * 60}")
print(f"DiffDock: {len(receptor_pdbs)} receptors × {len(ligand_sdfs)} ligands "
      f"= {len(receptor_pdbs) * len(ligand_sdfs)} combinations")
print(f"  batch_mode=True, batch_size={dd_cfg['batch_size']}")
print(f"{'═' * 60}\n")

results = run_diffdock(
    proteins=receptor_pdbs,
    ligands=ligand_sdfs,
    output_dir=output_base,
    cfg=dd_cfg,
    gpu_model=gpu_model,
)

# ── Save summary ────────────────────────────────────────────────────────────
summary = generate_summary(results, dd_cfg)
with open(output_base / "docking_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

n_ok   = sum(1 for r in results if r.status == "success")
n_fail = sum(1 for r in results if r.status == "failed")
n_skip = sum(1 for r in results if r.status == "skipped")

print(f"\n{'=' * 60}")
print(f"ORAI × BENCHMARK DIFFDOCK COMPLETE")
print(f"  Receptors:    {len(receptor_pdbs)}")
print(f"  Ligands:      {len(ligand_sdfs)}")
print(f"  Combinations: {len(results)}")
print(f"  Success:      {n_ok}")
print(f"  Failed:       {n_fail}")
print(f"  Skipped:      {n_skip}")
print(f"  Results dir:  {output_base}")
print(f"  Summary:      {output_base / 'docking_summary.json'}")
print(f"{'=' * 60}")

GPU: NVIDIA GeForce RTX 4070 Laptop GPU

Receptors (4):
  • Orai1WT-MDSnap-Fr300.pdb
  • Orai1WT-MDSnap-Fr400.pdb
  • Orai1WT-MDSnap-Fr499.pdb
  • Orai1WT-START-Fr0.pdb

Ligands: 428

════════════════════════════════════════════════════════════
DiffDock: 4 receptors × 428 ligands = 1712 combinations
  batch_mode=True, batch_size=15
════════════════════════════════════════════════════════════

Error log cleared (overwrite_error_log=true)
DiffDock Docking
Mode: CSV batch
Output directory: Dockings/Orai_Benchmark_DiffDock
Proteins: 4  |  Ligands: 428  |  Total: 1712
Previously failed (will skip): 0
Samples: 10  |  Steps: 20  |  Timeout: 600s
Pre-computing ligand properties...
  428 ligands analysed
Pre-computing protein properties...
  4 proteins analysed
Docking log: Dockings/Orai_Benchmark_DiffDock/docking_log.csv (375 existing entries)

Preparing inputs...
  Prepared 4 proteins (cached in prepared_proteins/)
  Prepared 428 ligands (cached in prepared_ligands/)

Skipping 389 already doc

# Orai × PoseBuster Benchmark Ligands || EquiBind (batch)

Docks every benchmark ligand (`*_ligand_start_conf.sdf`) against every Orai
receptor (`Orai*_cleaned.pdb`) using `run_equibind.py` in a single batch
invocation. Uses pre-computed fpocket + p2rank pockets for guided docking.


In [ ]:
import os, sys, json, subprocess, yaml
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

# ── Paths ────────────────────────────────────────────────────────────────────
benchmark_dir = Path("Data/PoseBuster Benchmark Set")
fpocket_dir   = Path("pocket_results/fpocket_results")           # source of *_cleaned.pdb
p2rank_dir    = Path("pocket_results/p2rank_results")
output_base   = Path("Dockings/Orai_Benchmark_Equibind")
log_dir       = Path("Dockings/Logs/orai_benchmark_equibind_logs")
config_path   = Path("Scripts/Docking/equibind_docking_config.yaml")
runner_script = Path("Scripts/Docking/run_equibind.py")
staging_root  = output_base / "_staging"
rec_staging   = staging_root / "receptors"
lig_staging   = staging_root / "ligands"

for d in (output_base, log_dir, rec_staging, lig_staging):
    d.mkdir(parents=True, exist_ok=True)

# ── Load EquiBind config (for equibind_dir / device / batch) ────────────────
with open(config_path) as f:
    eb_cfg = yaml.safe_load(f)

equibind_dir   = os.path.expanduser(eb_cfg.get("equibind_dir", "~/tools/EquiBind"))
device         = eb_cfg.get("device", "cuda")
gpu_batch_size = int(eb_cfg.get("gpu_batch_size", 8))

# Use EquiBind conda env's Python (notebook kernel lacks torch)
equibind_python = Path("/home/manndo/anaconda3/envs/equibind/bin/python")
if not equibind_python.exists():
    raise FileNotFoundError(f"EquiBind python not found: {equibind_python}")

print(f"EquiBind dir:    {equibind_dir}")
print(f"EquiBind python: {equibind_python}")
print(f"Device:          {device}")
print(f"GPU batch size:  {gpu_batch_size}")

# ── Stage receptors ──────────────────────────────────────────────────────────
# Use the *_cleaned.pdb files in fpocket_results so the protein name matches
# the pocket filenames (fpocket: <stem>_out/, p2rank: <stem>.pdb_predictions.csv).
receptor_pdbs = sorted(p for p in fpocket_dir.glob("Orai*_cleaned.pdb") if p.is_file())
if not receptor_pdbs:
    raise FileNotFoundError(f"No Orai *_cleaned.pdb receptors found in {fpocket_dir}")
for r in receptor_pdbs:
    link = rec_staging / r.name
    if not link.exists():
        link.symlink_to(r.resolve())
print(f"\nReceptors staged: {len(receptor_pdbs)}")
for r in receptor_pdbs:
    print(f"  • {r.name}")

# Sanity-check pockets exist for each receptor
missing_pockets = []
for r in receptor_pdbs:
    stem = r.stem  # e.g. "Orai1WT-MDSnap-Fr300_cleaned"
    fp_ok = (fpocket_dir / f"{stem}_out").is_dir()
    p2_ok = any(p2rank_dir.glob(f"{stem}*_predictions.csv"))
    if not (fp_ok and p2_ok):
        missing_pockets.append((r.name, fp_ok, p2_ok))
if missing_pockets:
    print("\n⚠ Receptors with missing pockets (fpocket, p2rank):")
    for name, fp, p2 in missing_pockets:
        print(f"  • {name}: fpocket={fp} p2rank={p2}")

# ── Stage benchmark ligands (symlink every *_ligand_start_conf.sdf) ─────────
complex_dirs = sorted(
    d for d in benchmark_dir.iterdir()
    if d.is_dir() and not d.name.startswith(("_", "."))
)
n_staged, n_missing = 0, 0
for cdir in complex_dirs:
    lig = cdir / f"{cdir.name}_ligand_start_conf.sdf"
    if not lig.exists():
        n_missing += 1
        continue
    link = lig_staging / lig.name
    if not link.exists():
        link.symlink_to(lig.resolve())
    n_staged += 1
print(f"\nBenchmark ligands staged: {n_staged}  (missing: {n_missing})")

n_combos = len(receptor_pdbs) * n_staged
print(f"\n{'═' * 60}")
print(f"EquiBind: {len(receptor_pdbs)} receptors × {n_staged} ligands = {n_combos} combinations")
print(f"  GPU batch size: {gpu_batch_size}")
print(f"{'═' * 60}\n")

# ── Invoke run_equibind.py once (it iterates all receptor×ligand pairs) ─────
env = os.environ.copy()
env.update({
    "EQ_RECEPTORS_DIR":   str(rec_staging.resolve()),
    "EQ_DRUGS_DIR":       str(lig_staging.resolve()),
    "EQ_OUTPUT_DIR":      str(output_base.resolve()),
    "EQ_RECEPTOR_FILTER": "_cleaned",                              # match *_cleaned.pdb
    "EQ_EQUIBIND_DIR":    equibind_dir,
    "EQ_DEVICE":          device,
    "EQ_GPU_BATCH_SIZE":  str(gpu_batch_size),
    "EQ_FPOCKET_DIR":     str(fpocket_dir.resolve()),
    "EQ_P2RANK_DIR":      str(p2rank_dir.resolve()),
})

log_file = log_dir / "orai_benchmark_equibind.log"
print(f"Logging to: {log_file}")
print(f"Output to:  {output_base}\n")

with open(log_file, "w") as lf:
    proc = subprocess.run(
        [str(equibind_python), str(runner_script.resolve())],
        env=env,
        cwd=str(Path.cwd()),
        stdout=lf,
        stderr=subprocess.STDOUT,
    )

# ── Report ──────────────────────────────────────────────────────────────────
summary_file = output_base / "pipeline_summary.json"
print(f"\n{'=' * 60}")
print(f"ORAI × BENCHMARK EQUIBIND COMPLETE")
print(f"{'=' * 60}")
print(f"  Return code:  {proc.returncode}")
print(f"  Log file:     {log_file}")
print(f"  Results dir:  {output_base}")

if proc.returncode == 0 and summary_file.exists():
    with open(summary_file) as f:
        s = json.load(f)
    totals = s.get("totals", {})
    timing = s.get("global_timing", {})
    print(f"  Poses ok:     {totals.get('poses_success', '?')}")
    print(f"  Poses failed: {totals.get('poses_failed', '?')}")
    print(f"  Wall time:    {timing.get('pipeline_wall_time_s', 0.0):.1f}s")
    print(f"  Summary:      {summary_file}")
else:
    print(f"  ✗ Failed — see log for details")
print(f"{'=' * 60}")


# Orai × PoseBuster Benchmark Ligands || PoseBusters validation (AutoDock + DiffDock + EquiBind)

Runs PoseBusters (`config="dock"`) on every Orai-vs-benchmark-ligand pose produced by all three docking methods.

**Receptor variants** (from `Data/Receptors/`):
- `Orai1WT-START-Fr0`
- `Orai1WT-MDSnap-Fr300` / `Fr400` / `Fr499`

**Source layouts consumed** (no renaming needed — the existing folder/file names already match the
`<protein>__<ligand>` / `<ligand>__<protein>` conventions used by the registered collectors):

| Method   | Path                                   | Pattern                                                     |
|----------|----------------------------------------|-------------------------------------------------------------|
| AutoDock | `Dockings/Orai_Benchmark/docking/`     | `<receptor>__<ligand>_vina_out.pdbqt`                       |
| DiffDock | `Dockings/Orai_Benchmark_DiffDock/`    | `<ligand>_start_conf__<receptor>/rank*.sdf`                 |
| EquiBind | `Dockings/Orai_Benchmark_Equibind/`    | `<ligand>_ligand_start_conf__<receptor>_cleaned/*.sdf`      |

**Staging done by this cell** (idempotent symlinks):
1. `posebusters_results/_orai_benchmark_staging/receptors/` — the 4 Orai PDBs only (so receptor discovery does not pick up unrelated PDBs in `Data/Receptors/`).
2. `posebusters_results/_orai_benchmark_staging/ligand_templates/` — per-ligand SDFs renamed to match the AutoDock ligand label, used as bond-order templates during PDBQT→SDF conversion.

Outputs go to `posebusters_results/orai_benchmark/dock/` (CSVs, plots, and a `posebuster_proved/dock/` copy of all passing poses).

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
receptors_src    = Path("Data/Receptors")
benchmark_dir    = Path("Data/PoseBuster Benchmark Set")
autodock_dir     = Path("Dockings/Orai_Benchmark/docking")
diffdock_dir     = Path("Dockings/Orai_Benchmark_DiffDock")
equibind_dir     = Path("Dockings/Orai_Benchmark_Equibind")

pb_runner_script = Path("Scripts/Docking/Posebusters/run_posebusters.py")
pb_config_path   = Path("Scripts/Docking/Posebusters/posebusters_orai_benchmark_config.yaml")

staging_root     = Path("posebusters_results/_orai_benchmark_staging")
rec_staging      = staging_root / "receptors"
tpl_staging      = staging_root / "ligand_templates"
for d in (rec_staging, tpl_staging):
    d.mkdir(parents=True, exist_ok=True)

# Restrict to the four Orai receptor variants (matches receptor names embedded
# in every docking output filename).
ORAI_RECEPTORS = [
    "Orai1WT-START-Fr0",
    "Orai1WT-MDSnap-Fr300",
    "Orai1WT-MDSnap-Fr400",
    "Orai1WT-MDSnap-Fr499",
]


def _link(src: Path, dst: Path) -> bool:
    if not src.exists():
        return False
    if dst.is_symlink() or dst.exists():
        try:
            if dst.resolve() == src.resolve():
                return True
        except OSError:
            pass
        dst.unlink()
    try:
        dst.symlink_to(src.resolve())
    except OSError:
        shutil.copy2(src, dst)
    return True


# ── 1. Stage receptors ──────────────────────────────────────────────────────
n_rec = 0
missing_rec = []
for name in ORAI_RECEPTORS:
    src = receptors_src / f"{name}.pdb"
    if _link(src, rec_staging / f"{name}.pdb"):
        n_rec += 1
    else:
        missing_rec.append(name)


# ── 2. Stage ligand templates (per-ligand SDF aliases) ──────────────────────
# AutoDock ligand label after collector parsing is "<pdb_id>_ligand_start_conf_vina"
# (filename minus "_vina_out", split on "__"). The bond-order template lookup
# in run_posebusters.py uses glob "*<ligand_label>*.sdf", so we expose each
# benchmark SDF under a name that contains that exact substring.
n_tpl = 0
ligand_labels_seen = set()
if benchmark_dir.is_dir():
    for cdir in sorted(benchmark_dir.iterdir()):
        if not cdir.is_dir() or cdir.name.startswith(("_", ".")):
            continue
        pdb_id = cdir.name
        sdf = cdir / f"{pdb_id}_ligand.sdf"
        if not sdf.exists():
            continue
        # Alias matching the AutoDock ligand label
        ad_label = f"{pdb_id}_ligand_start_conf_vina"
        if _link(sdf, tpl_staging / f"{ad_label}.sdf"):
            n_tpl += 1
            ligand_labels_seen.add(pdb_id)
        # Also expose the bare-id name (covers DiffDock / EquiBind label "<pdb_id>_(ligand_)?start_conf")
        _link(sdf, tpl_staging / f"{pdb_id}.sdf")


# ── 3. Quick sanity counts on the docking source folders ────────────────────
n_ad = sum(1 for _ in autodock_dir.glob("*_vina_out.pdbqt")) if autodock_dir.is_dir() else 0
n_dd = sum(
    1 for d in diffdock_dir.iterdir()
    if d.is_dir() and "__" in d.name and any(d.glob("**/*.sdf"))
) if diffdock_dir.is_dir() else 0
n_eb = sum(
    1 for d in equibind_dir.iterdir()
    if d.is_dir() and "__" in d.name
    and any(f for f in d.glob("**/*.sdf") if "prep" not in f.parts)
) if equibind_dir.is_dir() else 0

print("Staging summary")
print("─" * 60)
print(f"  receptors staged:   {n_rec:>5}  → {rec_staging}")
print(f"  ligand templates:   {n_tpl:>5}  → {tpl_staging}")
print(f"  AutoDock pose files (source): {n_ad:>5}  ({autodock_dir})")
print(f"  DiffDock pose dirs  (source): {n_dd:>5}  ({diffdock_dir})")
print(f"  EquiBind pose dirs  (source): {n_eb:>5}  ({equibind_dir})")
if missing_rec:
    print(f"\n⚠ Missing receptor PDB(s): {missing_rec}")

# ── 4. Sanity-check config + runner exist ───────────────────────────────────
if not pb_config_path.exists():
    raise FileNotFoundError(pb_config_path)
if not pb_runner_script.exists():
    raise FileNotFoundError(pb_runner_script)

# ── 5. Invoke run_posebusters.py with the Orai × Benchmark config ───────────
print(f"\n{'═' * 60}")
print(f"PoseBusters validation — Orai × PoseBuster Benchmark Set")
print(f"  config:  {pb_config_path}")
print(f"  runner:  {pb_runner_script}")
print(f"{'═' * 60}\n")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

proc = subprocess.run(
    [sys.executable, str(pb_runner_script.resolve()),
     "--config", str(pb_config_path.resolve())],
    cwd=str(Path.cwd()),
    env=env,
)

print(f"\n{'=' * 60}")
print(f"PoseBusters Orai × BENCHMARK COMPLETE  (return code: {proc.returncode})")
print(f"  Results: posebusters_results/orai_benchmark/dock/")
print(f"  Plots:   posebusters_results/orai_benchmark/dock/pb_*.png")
print(f"  Proved:  posebuster_proved/dock/")
print(f"{'=' * 60}")
